# Task A — MediaPipe tracker quality audit (Kaggle)

Run this **after** your Stage 1 v3 kernel has been committed (so its `stage1v3_results.json` and `skeleton_features_t32.pt` are available as inputs).

Two run modes:
1. **cache mode (fast, default, ~10 s)** — reads the existing skeleton cache.  Adequate for the headline Pearson(visibility, CER) and Pearson(dropout, CER) verdict.
2. **reextract mode (slow, ~30 min)** — re-streams the WiTA zips for the 16 audited signers and runs MediaPipe natively, producing 5 overlay GIFs per signer plus native-resolution per-frame metrics.

Kaggle does not need a GPU for this task — CPU is fine.  Turn off GPU in the kernel settings to save quota.

## Attach inputs on Kaggle (before running)

1. Open the kernel's right-hand panel → **Data → Add Data**.
2. Add your **Stage 1 v3 kernel's output** as a notebook-output dataset.  Kaggle mounts it at `/kaggle/input/<your-stage1v3-kernel-name>/`.
3. Confirm the two files exist at that mount:
   - `stage1v3_results.json`
   - `skeleton_features_t32.pt`

If you ran Stage 1 v3 in this same kernel and never restarted, the files are already in `/kaggle/working/` and you can skip step 2.

## Cell 1 — Install + clone

In [ ]:
%%capture
!pip install editdistance huggingface_hub hgtk 'mediapipe>=0.10.0' scipy imageio --quiet

import sys, os
!rm -rf /kaggle/working/wita_v2
!git clone -b iterative-ablation "https://github.com/Gaurs86/WiTA-v2.git" '/kaggle/working/wita_v2'
sys.path.insert(0, '/kaggle/working')

## Cell 2 — Locate the Stage 1 v3 artifacts

Edit the two paths below if your Stage 1 v3 kernel-output dataset is mounted under a different name.

In [ ]:
import os, glob

# Prefer files already in /kaggle/working/ (same-session run); fall back to the
# attached notebook-output dataset at /kaggle/input/.
def _first_existing(*candidates):
    for c in candidates:
        if c and os.path.exists(c):
            return c
    return None

RESULTS_PATH = _first_existing(
    '/kaggle/working/logs/stage1v3_results.json',
    '/kaggle/working/stage1v3_results.json',
    *glob.glob('/kaggle/input/*/logs/stage1v3_results.json'),
    *glob.glob('/kaggle/input/*/stage1v3_results.json'),
)
CACHE_PATH = _first_existing(
    '/kaggle/working/skeleton_features_t32.pt',
    *glob.glob('/kaggle/input/*/skeleton_features_t32.pt'),
)
OUTPUT_DIR = '/kaggle/working/tracker_audit'
MANIFEST   = '/kaggle/working/audit_clips.json'

assert RESULTS_PATH, 'stage1v3_results.json not found — attach the Stage 1 v3 kernel output.'
assert CACHE_PATH,   'skeleton_features_t32.pt not found — attach the Stage 1 v3 kernel output.'
print('results path :', RESULTS_PATH)
print('cache path   :', CACHE_PATH)
print('output dir   :', OUTPUT_DIR)

## Cell 3 — Run the audit (cache mode)

Reads the existing T_native=32 cache, computes per-clip quality metrics, joins with per-signer CER from `stage1v3_results.json`, and emits `per_signer_quality.csv` + `quality_vs_cer.png` + an `audit_summary.json` with the Pearson verdict.

In [ ]:
!python /kaggle/working/wita_v2/scripts/audit_tracker_quality.py \
    --stage1v3-results {RESULTS_PATH} \
    --cache-path       {CACHE_PATH} \
    --output-dir       {OUTPUT_DIR} \
    --audit-manifest   {MANIFEST} \
    --mode             cache

## Cell 4 — Inspect the result

Print the per-signer CSV head and show the scatter plot inline.

In [ ]:
import os, pandas as pd
from IPython.display import Image, display

df = pd.read_csv(os.path.join(OUTPUT_DIR, 'per_signer_quality.csv'))
df = df.sort_values('val_cer', ascending=False)
display(df)
display(Image(filename=os.path.join(OUTPUT_DIR, 'quality_vs_cer.png')))

## Cell 5 — (Optional) reextract mode for native-resolution metrics + overlay GIFs

Only run this if Cell 3's verdict was `tracker_partial` and you want to inspect the worst-signer trajectories by eye, OR if you want the native (pre-resample) dropout pattern.

Wall-clock: ~30 min on Kaggle CPU.  Network: re-downloads the WiTA zips for 16 audited signers (~3 GB peak before they're deleted post-extraction).

In [ ]:
# Uncomment to run.
# !python /kaggle/working/wita_v2/scripts/audit_tracker_quality.py \
#     --stage1v3-results {RESULTS_PATH} \
#     --output-dir       {OUTPUT_DIR} \
#     --audit-manifest   {MANIFEST} \
#     --max-overlays     5 \
#     --mode             reextract

## Cell 6 — Verdict and next step

The script's stdout in Cell 3 prints the decision (`tracker_dominant` / `tracker_null` / `tracker_partial`).  That decides one bit in `configs/stage3.yaml`:
- **tracker_dominant** (Pearson(visibility, CER) ≤ −0.5 or Pearson(dropout, CER) ≥ +0.5): keep `visibility_gate: true` (current default).
- **tracker_null** (|r| < 0.3 on all three): flip `visibility_gate: false` in Cell 2 of `run_stage3_cv.ipynb` and rebuild the cache with one fewer input dim.
- **tracker_partial**: inspect the overlay GIFs (Cell 5) and decide by eye.

The `audit_summary.json` in `/kaggle/working/tracker_audit/` archives the full verdict for the dissertation.